# Lesson 28 — Bilevel and Complementarity Programs

## Learning objectives

1. Write a bilevel optimization problem and recognize its structure.
2. Reformulate a convex-lower-level bilevel as MPCC / MPEC via KKT.
3. Use big-M relaxations of complementarity constraints.
4. Recognize Stackelberg games and toll-setting as bilevel.

## 1. Bilevel structure

$$
\begin{aligned}
\min_{x, y}\quad & F(x, y) \\
\text{s.t.}\quad & G(x, y) \le 0 \\
& y \in \arg\min_{y'} \{ f(x, y') : g(x, y') \le 0 \}.
\end{aligned}
$$

Outer problem (leader): $x$. Inner problem (follower): $y$ depends on $x$. Examples: pricing where competitors react, defender-attacker games, network design with user-equilibrium routing {cite:p}`Dempe2002,LuoPangRalph1996`.

Bilevel is *strictly harder* than the union of two NLPs — even with linear inner and outer it is NP-hard.

## 2. KKT reformulation

If the inner problem is convex with strong duality (Slater holds), replace $y \in \arg\min$ with the inner KKT conditions. With $g : \mathbb{R}^{n+m} \to \mathbb{R}^p$ and Jacobian $\nabla_y g(x, y) \in \mathbb{R}^{p \times m}$:

$$\nabla_y f(x, y) + \nabla_y g(x, y)^\top \lambda = 0, \quad g(x, y) \le 0, \quad \lambda \ge 0, \quad \lambda \circ g(x, y) = 0.$$

The complementarity $\lambda \circ g = 0$ is the troublesome bit — it's nonconvex (a union of $2^p$ faces depending on which inequalities are active). The reformulated problem is a **Mathematical Program with Complementarity Constraints (MPCC)** {cite:p}`LuoPangRalph1996`.

## 3. Solving MPCC

Three approaches:

- **Big-M / disjunctive MILP**: introduce binary $z_i$, add $\lambda_i \le M z_i$, $-g_i \le M(1 - z_i)$. Tight bounds critical.
- **Penalty / regularization**: $\lambda^\top g \le \epsilon$ for shrinking $\epsilon$.
- **Active-set**: enumerate $\lambda_i > 0$ vs $g_i < 0$ partitions.

Specialized solvers: PATH, MILES (mixed complementarity); IPOPT with NLP-relaxation often works for small MPCC.

## 4. Stackelberg pricing

Leader sets prices $p$; follower (consumer) chooses quantity $q(p)$ minimizing cost. Leader maximizes revenue $p \cdot q(p)$. The follower's best-response problem is convex; its KKT becomes the bilevel constraint.

Big-M reformulation gives a MILP for piecewise-linear demand.

## 5. Network toll-setting

You set tolls on a road network; users follow Wardrop equilibrium (each driver picks shortest-cost path). Maximize revenue subject to user equilibrium.

User-equilibrium → variational inequality → KKT → MPCC. Solved at scale only for moderate networks; instances with $n \sim 10^3$ links are challenging.

## References
{cite:p}`Dempe2002,LuoPangRalph1996`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Toy bilevel: leader picks tax t in [0, 5]; follower picks q in [0, 10]
# minimizing (q - 4)^2 + t * q.
# Leader maximizes revenue t * q*(t).
# Inner KKT: 2(q - 4) + t = 0 -> q* = 4 - t/2 (if positive).
# So leader: max_t t * (4 - t/2). Optimum t* = 4, q* = 2, revenue = 8.

m = Model("bilevel_toy")
t = m.continuous("t", lb=0, ub=5)
q = m.continuous("q", lb=0)
lam = m.continuous("lam", lb=0)        # KKT multiplier for q >= 0
m.subject_to(2*(q - 4) + t - lam == 0)    # inner stationarity
m.subject_to(lam * q == 0)                # complementarity (nonconvex!)
m.maximize(t * q)
r = m.solve()
print("leader's tax:", r.value(t), " follower q:", r.value(q), " revenue:", r.objective)
